In [ ]:
%pip install langgraph langchain-openai langchain dotenv arize-phoenix-otel openinference-instrumentation-langchain

In [1]:
from langgraph.prebuilt import create_react_agent
import dotenv
import uuid
from langchain_openai import ChatOpenAI
import requests



In [2]:
dotenv.load_dotenv()

True

In [3]:
from phoenix.otel import register
from openinference.instrumentation import using_metadata

# configure the Phoenix tracer
tracer_provider = register(
  project_name="pydata-seattle-2025-workshop", # Default is 'default'
  auto_instrument=True # Auto-instrument your app based on installed OI dependencies
)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: pydata-seattle-2025-workshop
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: 34.11.200.211:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [4]:
user_id = uuid.uuid4()


In [5]:
print(f"User ID: {user_id}  ")

User ID: e342fc14-142c-4305-94ed-f32b6348d298  


In [6]:
default_metadata = {
    "user_id": user_id,
    
}

In [7]:
print (f"""Add this line to filter traces 

metadata['user_id'] == "{user_id}"

""")

Add this line to filter traces 

metadata['user_id'] == "e342fc14-142c-4305-94ed-f32b6348d298"




In [8]:
def get_weather(city: str) -> str:  
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

In [9]:
agent = create_react_agent(
    model="openai:gpt-4o-mini",   
    tools=[get_weather],  
    prompt="You are a helpful assistant"  
)

In [18]:
with using_metadata(default_metadata):
    # Run the agent
    res = agent.invoke(
        {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
    )
res

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='15d1ad98-fa9d-4a35-80de-b8db345cb37c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Vbbqq0CzI3apElo2OftubAoo', 'function': {'arguments': '{"city":"San Francisco"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 56, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CQQc4yzPEDMbG6CCEErCK6aUfMMnZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--f9fe97e9-e929-4d6f-80f0-13a2feb67d36-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 

In [20]:
BASE = "http://localhost:8080"

llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=f"{BASE}/v1",
    temperature=0.2,
    max_tokens=512,
)

llm.invoke("what is the weather in sf")




AIMessage(content="I don't have real-time data access to provide current weather conditions. However, you can easily check the weather in San Francisco by using a weather website, app, or a search engine. If you need information about typical weather patterns in San Francisco or historical data, feel free to ask!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 13, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CQQdziMWUsS3h7PNSO3h0WruicnSo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--7fafc3d6-f6d8-4698-86c9-ca9f30eb0109-0', usage_metadata={'input_tokens': 13, 'output_tokens': 57, 'total_tokens': 70, 'input_token_d

In [22]:
def tavily_search(query, **kw):
    r = requests.post(f"{BASE}/v1/tavily/search",
                      json={"query": query, **kw}, timeout=60)
    r.raise_for_status()
    return r.json()

res = tavily_search("What is Langraph?")
print(res)

{'query': 'What is Langraph?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.ibm.com/think/topics/langgraph', 'title': 'What is LangGraph? - IBM', 'content': '*   [Overview](https://www.ibm.com/think/topics/ai-agents#7281535) *   [Overview](https://www.ibm.com/think/topics/components-of-ai-agents#498277090) *   [Learning](https://www.ibm.com/think/topics/ai-agent-learning#498277087) *   [Tutorial: LangGraph ReAct agent](https://www.ibm.com/think/tutorials/deploy-langgraph-react-agent-manage-it-support-tickets-watsonx-ai#1287801557) *   [Overview](https://www.ibm.com/think/topics/ai-agent-protocols#1509394340) *   [Tutorial: LangGraph ReAct agent](https://www.ibm.com/think/tutorials/deploy-langgraph-react-agent-manage-it-support-tickets-watsonx-ai#80364620) *   [Overview](https://www.ibm.com/think/insights/ai-agent-governance#1268897081) *   [Overview](https://www.ibm.com/think/topics/ai-agent-use-cases#257779831) *   [Human resources](https

## References

https://arize.com/docs/phoenix/integrations/python/langgraph/langgraph-tracing